In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

# Set environment to use local model paths
os.environ['HF_HOME'] = '/net/projects/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects/chai-lab/shared_models'

repo_path = "/net/scratch2/smallyan/filter_eval"
os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

Working directory: /net/scratch2/smallyan/filter_eval


# Code Evaluation for Circuit Analysis (Filter Heads Project)

Repository: `/net/scratch2/smallyan/filter_eval`

This notebook evaluates the code implementation for the "LLMs Process Lists With General Filter Heads" project.

## 1. Setup and Configuration

In [2]:
# Initialize evaluation tracking
evaluation_results = []

def add_eval_result(cell_id, description, runnable, correct_impl, redundant, irrelevant, notes=""):
    """Helper function to add evaluation results"""
    evaluation_results.append({
        "cell_id": cell_id,
        "description": description,
        "runnable": runnable,
        "correct_implementation": correct_impl,
        "redundant": redundant,
        "irrelevant": irrelevant,
        "notes": notes
    })
    status = "PASS" if runnable == "Y" and correct_impl == "Y" else "ISSUE"
    print(f"[{status}] {cell_id}: Runnable={runnable}, Correct={correct_impl}")
    if notes:
        print(f"   Notes: {notes}")

# Track corrected blocks
corrected_blocks = []
failed_blocks = []

print("Evaluation framework initialized")

Evaluation framework initialized


In [3]:
# Check GPU and environment
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA A100 80GB PCIe
Memory: 85.1 GB


## 2. Evaluating demo.ipynb

The demo notebook is the main entry point according to CodeWalkthrough.md. It demonstrates:
1. Loading the model
2. Verifying filter head behavior
3. Patching query states to transfer predicates
4. Testing multiple filter heads

In [4]:
# Cell 0: autoreload magic commands
add_eval_result(
    cell_id="demo.ipynb:Cell_0",
    description="Autoreload magic commands",
    runnable="Y",
    correct_impl="Y",
    redundant="N",
    irrelevant="N",
    notes="Jupyter magic for development"
)

[PASS] demo.ipynb:Cell_0: Runnable=Y, Correct=Y
   Notes: Jupyter magic for development


In [5]:
# Cell 1: Import and model loading
# Testing with local model path for Gemma-2-27b-it

try:
    import torch
    import transformers
    from src.models import ModelandTokenizer

    print(f"torch version: {torch.__version__}")
    print(f"transformers version: {transformers.__version__}")

    # Use local path for Gemma model
    model_key = "/net/projects/chai-lab/shared_models/gemma-2-27b-it"
    
    mt = ModelandTokenizer(
        model_key=model_key,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    
    print(f"Model loaded: {mt.model_key}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_1",
        description="Import libraries and load model",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="Model loaded successfully from local path"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_1")
    add_eval_result(
        cell_id="demo.ipynb:Cell_1",
        description="Import libraries and load model",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )
    raise

/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


`torch_dtype` is deprecated! Use `dtype` instead!


torch version: 2.7.1+cu118
transformers version: 4.57.3


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/12 [00:00<?, ?it/s]

[ISSUE] demo.ipynb:Cell_1: Runnable=N, Correct=Y
   Notes: Error: 'Gemma2ForCausalLM' object has no attribute 'model_key'


AttributeError: 'Gemma2ForCausalLM' object has no attribute 'model_key'

In [6]:
# Cell 1 - correct evaluation (model loaded successfully, just accessed wrong attribute)
# The model is loaded, just the demo code uses model_key but class uses .name
print(f"Model name: {mt.name}")
print(f"Model n_layer: {mt.n_layer}")
print(f"Model n_embd: {mt.n_embd}")

# Update the evaluation - the model loading works, the print statement in demo just had inconsistent attribute name
evaluation_results[-1] = {
    "cell_id": "demo.ipynb:Cell_1",
    "description": "Import libraries and load model",
    "runnable": "Y",
    "correct_implementation": "Y",
    "redundant": "N",
    "irrelevant": "N",
    "notes": "Model loaded successfully; mt.model_key should be mt.name (minor inconsistency)"
}
print("[PASS] demo.ipynb:Cell_1: Model loaded correctly")

Model name: /net/projects/chai-lab/shared_models/gemma-2-27b-it
Model n_layer: 46
Model n_embd: 4608
[PASS] demo.ipynb:Cell_1: Model loaded correctly


In [7]:
# Cell 2: Select filter head based on model
# For Gemma-2-27b-it, the code specifies head (29, 3)

try:
    model_key = "google/gemma-2-27b-it"  # This is what the demo expects
    
    if model_key == "meta-llama/Llama-3.3-70B-Instruct":
        layer_idx, head_idx = 35, 19
    elif model_key == "google/gemma-2-27b-it":
        layer_idx, head_idx = 29, 3
    else:
        raise ValueError("For other models you need to localize the heads first.")
    
    print(f"Selected filter head: Layer {layer_idx}, Head {head_idx}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_2",
        description="Select filter head based on model type",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="Filter head selection logic works correctly"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_2")
    add_eval_result(
        cell_id="demo.ipynb:Cell_2",
        description="Select filter head based on model type",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )

Selected filter head: Layer 29, Head 3
[PASS] demo.ipynb:Cell_2: Runnable=Y, Correct=Y
   Notes: Filter head selection logic works correctly


In [8]:
# Cell 4: Load SelectOneTask data
try:
    from src.selection.data import SelectOneTask
    from typing import Literal
    import os

    ##########################################################
    prompt_template_idx = 3  # try out different templates
    option_style: Literal["single_line", "numbered"] = "single_line"
    n_distractors = 5  # number of distractors
    ##########################################################

    select_task = SelectOneTask.load(
        path=os.path.join(
            "data_save", 
            "selection", 
            "objects.json"
        )
    )
    
    print(f"Task loaded: {type(select_task).__name__}")
    print(f"Categories available: {list(select_task.categories.keys())[:5]}...")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_4",
        description="Load SelectOneTask data",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="SelectOneTask data loaded successfully"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_4")
    add_eval_result(
        cell_id="demo.ipynb:Cell_4",
        description="Load SelectOneTask data",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )
    import traceback
    traceback.print_exc()

['name', 'prompt_templates', 'odd_one_prompt_templates', 'order_prompt_templates', 'count_prompt_templates', 'yes_no_prompt_templates', 'first_item_in_cat_prompt_templates', 'last_item_in_cat_prompt_templates', 'categories', 'exclude_categories']
Task loaded: SelectOneTask
[ISSUE] demo.ipynb:Cell_4: Runnable=N, Correct=Y
   Notes: Error: 'list' object has no attribute 'keys'


Traceback (most recent call last):
  File "/tmp/ipykernel_2741347/2214674547.py", line 22, in <module>
    print(f"Categories available: {list(select_task.categories.keys())[:5]}...")
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'list' object has no attribute 'keys'


In [9]:
# The task loaded correctly, just my inspection code was wrong
# Let me check the actual structure
print(f"Task type: {type(select_task)}")
print(f"Categories type: {type(select_task.categories)}")
print(f"Categories sample: {select_task.categories[:3] if isinstance(select_task.categories, list) else 'not a list'}")

# The task actually loaded correctly - my print statement was wrong
evaluation_results[-1] = {
    "cell_id": "demo.ipynb:Cell_4",
    "description": "Load SelectOneTask data",
    "runnable": "Y",
    "correct_implementation": "Y",
    "redundant": "N",
    "irrelevant": "N",
    "notes": "SelectOneTask data loaded successfully"
}
print("[PASS] demo.ipynb:Cell_4 - Task loaded correctly")

Task type: <class 'src.selection.data.SelectOneTask'>
Categories type: <class 'list'>
Categories sample: ['fruit', 'vehicle', 'furniture']
[PASS] demo.ipynb:Cell_4 - Task loaded correctly


In [10]:
# Cell 5: Get random sample
try:
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        category="fruit",
        filter_by_lm_prediction=True, 
    )

    print(f"Sample prompt: {sample.prompt()[:100]}...")
    print(f"Target object: {sample.obj}")
    print(f"Answer token: {mt.tokenizer.decode([sample.ans_token_id])}")
    
    add_eval_result(
        cell_id="demo.ipynb:Cell_5",
        description="Get random sample from task",
        runnable="Y",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes="Random sample generation works correctly"
    )
except Exception as e:
    failed_blocks.append("demo.ipynb:Cell_5")
    add_eval_result(
        cell_id="demo.ipynb:Cell_5",
        description="Get random sample from task",
        runnable="N",
        correct_impl="Y",
        redundant="N",
        irrelevant="N",
        notes=f"Error: {str(e)[:200]}"
    )
    import traceback
    traceback.print_exc()

[ISSUE] demo.ipynb:Cell_5: Runnable=N, Correct=Y
   Notes: Error: 'str' object is not callable


Traceback (most recent call last):
  File "/tmp/ipykernel_2741347/2935887578.py", line 3, in <module>
    sample = select_task.get_random_sample(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/selection/data.py", line 636, in get_random_sample
    is_correct, predictions, track_objs = verify_correct_option(
                                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/selection/utils.py", line 82, in verify_correct_option
    logits = get_hs(
             ^^^^^^^
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/functional.py", line 879, in get_hs
    with mt.trace(input, scan=False):
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/nnsight/contexts/Runner.py", line 41, in __exit__
    

In [11]:
# This seems to be an nnsight API issue with Gemma2 models
# The error occurs when trying to use module.output.save() with Gemma2

# Let me try a simpler approach - test if the basic model inference works
try:
    from src.tokens import prepare_input
    
    test_prompt = "Hello, how are you?"
    test_input = prepare_input(prompts=test_prompt, tokenizer=mt)
    print(f"Input prepared: {test_input.keys()}")
    
    # Try basic forward pass
    with torch.no_grad():
        outputs = mt._model(**test_input)
    print(f"Forward pass works: output shape = {outputs.logits.shape}")
    
except Exception as e:
    print(f"Basic inference error: {e}")
    import traceback
    traceback.print_exc()

Input prepared: KeysView({'input_ids': tensor([[     2,   4521, 235269,   1368,    708,    692, 235336]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]], device='cuda:0')})


Forward pass works: output shape = torch.Size([1, 7, 256000])


In [12]:
# The model works for basic inference. The issue is with nnsight tracing for Gemma2 models.
# This is likely a compatibility issue between nnsight and Gemma2ForCausalLM

# Let me try to use the nnsight trace directly
try:
    with mt.trace(test_input, scan=False):
        # Try to access a layer output
        layer_output = mt.model.layers[0].output.save()
    print(f"nnsight trace works: {layer_output.shape}")
except Exception as e:
    print(f"nnsight trace issue: {e}")
    
# Let me check the model structure
print(f"\nModel type: {type(mt._model)}")
print(f"Has sliding window: {hasattr(mt._model.config, 'sliding_window')}")

nnsight trace issue: 'str' object is not callable

Model type: <class 'transformers.models.gemma2.modeling_gemma2.Gemma2ForCausalLM'>
Has sliding window: True


In [13]:
# This is a known issue - Gemma2 uses a different architecture that may not be fully compatible
# with the current nnsight version. Let me document this as a special case.

# Record this as a runnable issue due to external dependency (nnsight + Gemma2 compatibility)
evaluation_results[-1] = {
    "cell_id": "demo.ipynb:Cell_5",
    "description": "Get random sample from task",
    "runnable": "N",
    "correct_implementation": "Y",
    "redundant": "N",
    "irrelevant": "N",
    "notes": "nnsight library incompatibility with Gemma2ForCausalLM model. Code logic is correct but external dependency issue."
}
failed_blocks.append("demo.ipynb:Cell_5")

# Since Gemma2 has nnsight issues, let me try to check if the code works with basic functionality
# by examining what the code is supposed to do without the nnsight tracing

print("Issue identified: nnsight library has compatibility issues with Gemma2ForCausalLM")
print("This is an external dependency issue, not a code implementation error.")
print("The code logic appears correct based on the implementation.")

Issue identified: nnsight library has compatibility issues with Gemma2ForCausalLM
This is an external dependency issue, not a code implementation error.
The code logic appears correct based on the implementation.


In [14]:
# Clear GPU memory and try with Llama model
import gc
del mt
gc.collect()
torch.cuda.empty_cache()
print("Cleared GPU memory")

Cleared GPU memory


In [15]:
# Load Llama-3.1-70B-Instruct
# This model is available locally and should work with nnsight

model_key = "/net/projects/chai-lab/shared_models/Meta-Llama-3.1-70B-Instruct"

mt = ModelandTokenizer(
    model_key=model_key,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

print(f"Model loaded: {mt.name}")
print(f"Model layers: {mt.n_layer}")

# For Llama 3.1, use the same filter heads as Llama 3.3 (similar architecture)
layer_idx, head_idx = 35, 19
print(f"Using filter head: Layer {layer_idx}, Head {head_idx}")

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded: /net/projects/chai-lab/shared_models/Meta-Llama-3.1-70B-Instruct
Model layers: 80
Using filter head: Layer 35, Head 19


In [16]:
# Now test Cell 5 again with Llama model
try:
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        category="fruit",
        filter_by_lm_prediction=True, 
    )

    print(f"Sample prompt: {sample.prompt()[:100]}...")
    print(f"Target object: {sample.obj}")
    print(f"Answer token: {mt.tokenizer.decode([sample.ans_token_id])}")
    
    # Update evaluation for Cell 5
    evaluation_results[-1] = {
        "cell_id": "demo.ipynb:Cell_5",
        "description": "Get random sample from task",
        "runnable": "Y",
        "correct_implementation": "Y",
        "redundant": "N",
        "irrelevant": "N",
        "notes": "Works correctly with Llama model; Gemma2 has nnsight compatibility issues"
    }
    if "demo.ipynb:Cell_5" in failed_blocks:
        failed_blocks.remove("demo.ipynb:Cell_5")
        corrected_blocks.append("demo.ipynb:Cell_5")
    print("[PASS] demo.ipynb:Cell_5")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Error: 'str' object is not callable


Traceback (most recent call last):
  File "/tmp/ipykernel_2741347/1053571846.py", line 3, in <module>
    sample = select_task.get_random_sample(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/selection/data.py", line 636, in get_random_sample
    is_correct, predictions, track_objs = verify_correct_option(
                                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/selection/utils.py", line 82, in verify_correct_option
    logits = get_hs(
             ^^^^^^^
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/torch/utils/_contextlib.py", line 116, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/net/scratch2/smallyan/filter_eval/src/functional.py", line 879, in get_hs
    with mt.trace(input, scan=False):
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/nnsight/contexts/Runner.py", line 41, in __exit__
    

In [17]:
# Check nnsight version and try a simple trace
import nnsight
print(f"nnsight version: {nnsight.__version__}")

# Try a minimal nnsight trace to understand the issue
from src.models import get_module_nnsight

try:
    test_input = prepare_input(prompts="Hello world", tokenizer=mt)
    with mt.trace(test_input, scan=False):
        # Try getting the embedding layer output
        emb = mt.model.embed_tokens
        emb_out = emb.output.save()
    print(f"Embedding output shape: {emb_out.shape}")
except Exception as e:
    print(f"Minimal trace error: {e}")

AttributeError: module 'nnsight' has no attribute '__version__'

In [18]:
# Check the conda environment file to understand required versions
import yaml
with open(os.path.join(repo_path, "conda_env.yml"), 'r') as f:
    env_content = f.read()
print(env_content)

# TODO: prune the dependencies. definitely don't need all these packages.
#! `baukit` needs to installed separately with `pip install git+https://github.com/davidbau/baukit`

name: connection
channels:
  - conda-forge
  - defaults
  - https://repo.anaconda.com/pkgs/main
  - https://repo.anaconda.com/pkgs/r
dependencies:
  - _libgcc_mutex=0.1=main
  - _openmp_mutex=5.1=1_gnu
  - asttokens=3.0.0=pyhd8ed1ab_1
  - bzip2=1.0.8=h5eee18b_6
  - ca-certificates=2025.2.25=h06a4308_0
  - comm=0.2.2=pyhd8ed1ab_1
  - debugpy=1.8.14=py311hfdbb021_0
  - decorator=5.2.1=pyhd8ed1ab_0
  - exceptiongroup=1.2.2=pyhd8ed1ab_1
  - executing=2.2.0=pyhd8ed1ab_0
  - importlib-metadata=8.6.1=pyha770c72_0
  - ipykernel=6.29.5=pyh3099207_0
  - ipython=9.1.0=pyhfb0248b_0
  - ipython_pygments_lexers=1.1.1=pyhd8ed1ab_0
  - jedi=0.19.2=pyhd8ed1ab_1
  - jupyter_client=8.6.3=pyhd8ed1ab_1
  - jupyter_core=5.7.2=pyh31011fe_1
  - krb5=1.21.3=h143b758_0
  - ld_impl_linux-64=2.40=h12ee557_0
  - libedit=3.1.20230828=h5eee18b_

In [19]:
# The conda_env.yml specifies nnsight==0.5.2 but we have 0.2.21 installed
# This is likely the source of the incompatibility

# Let me check what version is installed
import subprocess
result = subprocess.run(['pip', 'show', 'nnsight'], capture_output=True, text=True)
print(result.stdout)

print("\nRequired version from conda_env.yml: nnsight==0.5.2")
print("This version mismatch is likely causing the issues")

Name: nnsight
Version: 0.2.21
Summary: Package for interpreting and manipulating the internals of deep learning models.
Home-page: https://github.com/ndif-team/nnsight
Author: 
Author-email: Jaden Fiotto-Kaufman <jadenfk@outlook.com>
License: 
Location: /home/smallyan/.local/lib/python3.12/site-packages
Requires: accelerate, diffusers, einops, protobuf, pydantic, python-socketio, sentencepiece, tokenizers, torch, torchvision, transformers
Required-by: leela-interp


Required version from conda_env.yml: nnsight==0.5.2
This version mismatch is likely causing the issues


In [20]:
# Install the correct version of nnsight
import subprocess
print("Installing nnsight==0.5.2...")
result = subprocess.run(['pip', 'install', 'nnsight==0.5.2'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f"Error: {result.stderr}")

Installing nnsight==0.5.2...


Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: nnsight
    Found existing installation: nnsight 0.2.21
    Uninstalling nnsight-0.2.21:
      Successfully uninstalled nnsight-0.2.21

